In [1]:
import pandas as pd

train = pd.read_csv('household_train.csv')
valid = pd.read_csv('household_validation.csv')
test = pd.read_csv('household_test.csv')

In [2]:
input_width = 60
label_width = 30

## Windowing

In [3]:
import numpy as np
import tensorflow as tf

def create_sequences(data, input_width, label_width, target_column="Global_active_power"):
    X, y = [], []
    values = data[target_column].values
    for i in range(len(values) - input_width - label_width):
        X.append(values[i:i+input_width])
        y.append(values[i+input_width:i+input_width+label_width])
    return np.array(X), np.array(y)

# Sequenzen erstellen
X_train, y_train = create_sequences(train, input_width, label_width)
X_valid, y_valid = create_sequences(valid, input_width, label_width)
X_test, y_test   = create_sequences(test,  input_width, label_width)

# CNN erwartet 3D-Input: (Samples, TimeSteps, Features)
X_train = X_train[..., np.newaxis]
X_valid = X_valid[..., np.newaxis]
X_test  = X_test[..., np.newaxis]


2025-08-28 20:35:19.716788: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## CNN

In [4]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

model = Sequential([
    Conv1D(filters=32, kernel_size=2, activation='relu', input_shape=(input_width, 1)),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(label_width)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()


/Users/basti/miniforge3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 59, 32)         │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 29, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 928)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        59,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 30)             │         1,950 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,502 (240.24 KB)

 Trainable params: 61,502 (240.24 KB)

 Non-trainable params: 0 (0.00 B)

## Training

In [5]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=10,
    batch_size=32
)


Epoch 1/10
50005/50005 ━━━━━━━━━━━━━━━━━━━━ 143s 3ms/step - loss: 0.4936 - mae: 0.4094 - val_loss: 0.4238 - val_mae: 0.3547
Epoch 2/10
50005/50005 ━━━━━━━━━━━━━━━━━━━━ 143s 3ms/step - loss: 0.4841 - mae: 0.4052 - val_loss: 0.4154 - val_mae: 0.3574
Epoch 3/10
50005/50005 ━━━━━━━━━━━━━━━━━━━━ 125s 2ms/step - loss: 0.4793 - mae: 0.4031 - val_loss: 0.4116 - val_mae: 0.3544
Epoch 4/10
50005/50005 ━━━━━━━━━━━━━━━━━━━━ 176s 4ms/step - loss: 0.4775 - mae: 0.4022 - val_loss: 0.4128 - val_mae: 0.3557
Epoch 5/10
50005/50005 ━━━━━━━━━━━━━━━━━━━━ 112s 2ms/step - loss: 0.4766 - mae: 0.4018 - val_loss: 0.4117 - val_mae: 0.3454
Epoch 6/10
50005/50005 ━━━━━━━━━━━━━━━━━━━━ 118s 2ms/step - loss: 0.4761 - mae: 0.4014 - val_loss: 0.4171 - val_mae: 0.3928
Epoch 7/10
50005/50005 ━━━━━━━━━━━━━━━━━━━━ 217s 4ms/step - loss: 0.4754 - mae: 0.4010 - val_loss: 0.4080 - val_mae: 0.3499
Epoch 8/10
50005/50005 ━━━━━━━━━━━━━━━━━━━━ 167s 3ms/step - loss: 0.4748 - mae: 0.4006 - val_loss: 0.4101 - val_mae: 0.3566
Epoch 9/

## Evaluation

In [6]:
test_loss, test_mae = model.evaluate(X_test, y_test)
print(f"Test MSE: {test_loss:.4f}, Test MAE: {test_mae:.4f}")


10792/10792 ━━━━━━━━━━━━━━━━━━━━ 15s 1ms/step - loss: 0.3291 - mae: 0.3556
Test MSE: 0.3291, Test MAE: 0.3556


## Prediction

In [7]:
y_pred = model.predict(X_test)
print(y_pred.shape)  


10792/10792 ━━━━━━━━━━━━━━━━━━━━ 11s 983us/step
(345333, 30)
